In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def load_train_data(path):
    df = pd.read_csv(path)
    df = df.apply(pd.to_numeric, errors="coerce") #To avoid NAN error; It converts empty space to nan
    df = df.dropna() # This drops the NAN data

    X = df.iloc[:, :-1].values.astype(float)
    y = df.iloc[:, -1].values.astype(float)
    return X, y


def load_test_data(path):
    df = pd.read_csv(path)
    df = df.apply(pd.to_numeric, errors="coerce") 
    df = df.dropna()
    return df.values.astype(float)

def normalize_train(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    sigma[sigma == 0] = 1
    X_norm = (X - mu) / sigma #z-score normalization
    return X_norm, mu, sigma

def normalize_test(X, mu, sigma):
    return (X - mu) / sigma

def polynomial_features(X, degree):
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    return np.hstack([X ** i for i in range(1, degree + 1)])

def compute_cost(X, y, w, b):
    m = X.shape[0]
    errors = (X @ w + b) - y
    return (errors @ errors) / (2 * m)

def compute_gradient(X, y, w, b):
    m = X.shape[0]
    errors = (X @ w + b) - y
    dj_dw = (X.T @ errors) / m
    dj_db = np.sum(errors) / m
    return dj_db, dj_dw

def polynomial_regression(X, y, degree=2, learning_rate=0.001, num_iters=1000):
    X_norm, mu, sigma = normalize_train(X)
    X_poly = polynomial_features(X_norm, degree)

    w = np.zeros(X_poly.shape[1])
    b = 0.0
    J_history = []

    for i in range(num_iters):
        dj_db, dj_dw = compute_gradient(X_poly, y, w, b)

        w -= learning_rate * dj_dw
        b -= learning_rate * dj_db

        if i % 50 == 0:
            cost = compute_cost(X_poly, y, w, b)
            J_history.append(cost)
            print(f"Iteration {i:4d}: Cost {cost:.6f}")

    return w, b, mu, sigma, J_history

def predict(X, w, b, mu, sigma, degree):
    X_norm = normalize_test(X, mu, sigma)
    X_poly = polynomial_features(X_norm, degree)
    return X_poly @ w + b

#To run the model
X_train, y_train = load_train_data("poly_train.csv")
X_test = load_test_data("poly_test.csv")
degree = 2
learning_rate = 0.001
num_iters = 1000
w, b, mu, sigma, J_history = polynomial_regression(X_train, y_train,degree=degree,learning_rate=learning_rate,num_iters=num_iters)
y_pred_test = predict(X_test, w, b, mu, sigma, degree)
print(y_pred_test[:5])


Iteration    0: Cost 2116045817228570890310017845057187650486221667033514302126240624956447546379599872.000000
Iteration   50: Cost 1575290537925763618716360888233930295915875712679687975044436900040483405429735424.000000
Iteration  100: Cost 1275794224309596260622115341129307201715423530342310815347816980724877127931920384.000000
Iteration  150: Cost 1085206662957606789917667359318076074776042395596642187739941619044654900404289536.000000
Iteration  200: Cost 949680418731549790796411639193218759352029687269091560819792140075597932951240704.000000
Iteration  250: Cost 846129416522828665167475931465925341076656118708479380717642773439803573959393280.000000
Iteration  300: Cost 763743459449029045510721774408402003809581506410175732547396513238403520189693952.000000
Iteration  350: Cost 696789145632710504125345056981262523739570756775704951593744270514522018176565248.000000
Iteration  400: Cost 641772681231388120715034394038881931591018788604195629408808360059944483048914944.000000
Iterat